# 1 TCP/IP Stack (Internet Protocol Suite)
— **Link layer**: communication within a single network segment (e.g., Ethernet, Wi-Fi).

— **Internet layer**: communication between networks; includes IP (IPv4/IPv6).

— **Transport layer**: communication between hosts; includes TCP (Transmission Control Protocol) and UDP (User Datagram Protocol).

— **Application layer**: data exchange between processes for applications.
<img src="https://upload.wikimedia.org/wikipedia/commons/thumb/c/c4/IP_stack_connections.svg/langru-800px-IP_stack_connections.svg.png" width="400">

# 2 HTTP Protocol

HTTP (Hypertext Transfer Protocol) is one of the main application-layer protocols.

- It is a stateless client-server protocol: the client (usually a web browser or a script) sends a request to the server, and the server returns a response.
- HTTP typically uses TCP as its transport protocol.

**HTTP/1.1 request:**
```http
GET /path/resource.html HTTP/1.1
Host: example.com
User-Agent: MyClient/1.0
Accept: text/html

```
An HTTP request has the following basic structure:

- **Request line**: method, resource path (URL), and HTTP version.
- **Headers**: metadata describing the request (e.g., Host, User-Agent, Accept).
- **Empty line**
- **Message body**: optional data.

The method (e.g., GET, POST, PUT, DELETE) defines the action. For example, GET retrieves a resource, POST sends data for processing (e.g., form submission). The path is the resource URI. Headers carry information such as content types or authorization. Data can be included in the request body (e.g., JSON or form data).

**HTTP/1.1 response:**
```http
HTTP/1.1 200 OK
Content-Type: text/html; charset=utf-8
Content-Length: 137

<html>...</html>
```
An HTTP response consists of:

- **Status line**: HTTP version, status code, and status message (e.g., HTTP/1.1 200 OK).
- **Headers**: (e.g., Content-Type, Content-Length).
- **Empty line**
- **Message body**: optional content (e.g., an HTML page, JSON data).

The status code indicates the result of request processing: 1xx — informational, 2xx — success, 3xx — redirection, 4xx — client error, 5xx — server error. For example, 200 OK means success, 404 Not Found means the resource does not exist. The body contains the returned resource (e.g., a web page or JSON).

**Header categories**
- **General**: apply to both requests and responses (e.g., `Cache-Control`, `Connection`)
- **Request headers**: `Host`, `User-Agent`, `Accept`, `Accept-Encoding`, `Authorization`, `If-None-Match`
- **Response headers**: `Server`, `Set-Cookie`, `WWW-Authenticate`
- **Entity headers**: `Content-Type`, `Content-Length`, `Content-Encoding`, `Content-Language`

**Header notes**
- The `Host` header is mandatory in HTTP/1.1.
- `Connection: close` and `Connection: keep-alive` control connection lifetime in HTTP/1.1.
- `Transfer-Encoding: chunked` allows streaming data of unknown size.
- `Content-Encoding`, `Accept-Encoding` (e.g., gzip) indicate the compression type.
- The `Range` header is used when requesting partial content (resumable downloads).

## Passing Data from Client to Server via URL
Encoding parameters in the URL is one of the primary ways to pass data in GET and POST HTTP requests.

Example URL for a GET request to search iTunes for records related to Radiohead:
- `https://itunes.apple.com/search?term=radiohead`
- This calls the `search` function with the parameter `term=radiohead`.

URLs can only contain a limited set of ASCII characters (letters, digits, and a few symbols). Other characters — such as spaces, punctuation, or non-Latin text — must be encoded to conform to URL format.

| Character | Encoded As               |
| --------- | ------------------------ |
| Space     | `%20`                    |
| `&`       | `%26`                    |
| `=`       | `%3D`                    |
| `?`       | `%3F`                    |
| `#`       | `%23`                    |
| `ü`       | `%C3%BC` (UTF-8 encoded) |

URL with encoded parameters:
- `https://itunes.apple.com/search?term=depeche%20mode`

URL with multiple parameters:
- `https://example.com/path?key1=value1&key2=value2`
- The part of the URL after `?` is called the **query string**; parameters are separated by `&`.

## Connection Management

**Persistent connections (HTTP/1.1)**
HTTP/1.0 connections were closed after every response; HTTP/1.1 introduced persistent connections (`Connection: keep-alive`), allowing multiple requests/responses to reuse a single TCP connection, reducing TCP/TLS overhead.

**Pipelining (HTTP/1.1)**
Clients can send multiple requests without waiting for responses, but responses must arrive in the same order.

**Multiplexing (HTTP/2)**
HTTP/2 allows data from multiple requests to be transmitted over a single TCP connection.

**HTTP/3 (QUIC)**
HTTP/3 runs on top of QUIC (built on UDP).

## Cookies and Sessions

**Cookies**
- The `Set-Cookie` response header instructs the client to store cookies.
- Cookies include attributes: `Path`, `Domain`, `Expires`/`Max-Age`, `HttpOnly`, `Secure`, `SameSite`.

**Sessions**
- Sessions are implemented by storing a session ID in a cookie (in the browser); the server uses this ID to restore the session context on subsequent requests.

Example of performing a GET request using the built-in HTTP client.

In [ ]:
import http.client
import json
from urllib.parse import urlparse

def http_client_get(url):
    p = urlparse(url)
    conn = http.client.HTTPSConnection(p.netloc, timeout=10)
    path = p.path or '/'
    if p.query:
        path += '?' + p.query
    conn.request('GET', path, headers={'User-Agent': 'py-http-client/1.0', 'Accept': '*/*'})
    resp = conn.getresponse()
    print(resp.status, resp.reason)
    for k,v in resp.getheaders():
        print(f'{k}: {v}')
    data = resp.read()
    conn.close()
    return data

print(http_client_get('https://www.google.com/').decode())

# 3 REST API
REST (Representational State Transfer) is a set of conventions for designing web APIs. RESTful resources (data objects such as users, records, etc.) are identified by URIs and manipulated using standard HTTP methods. For example, GET retrieves a resource, POST creates one, PUT or PATCH updates it, and DELETE removes it. REST emphasizes a uniform interface: clients interact with the server using consistent methods and resource representations (usually JSON).

REST (Representational State Transfer) is an architectural style for designing networked applications and APIs.
Key characteristics:
- A uniform interface for data access (resources are manipulated through a consistent set of operations).
- Uses HTTP methods and status codes for communication.
- The server exposes resources (data such as users, posts, tasks, etc.) and clients interact with them using standard HTTP methods.

HTTP methods commonly used in REST correspond to CRUD operations (Create-Read-Update-Delete):

- **GET** — request to retrieve a resource.
- **POST** — send data to the server to create a new resource. Often causes a server state change.
- **PUT** — update an existing resource by replacing it with the data provided in the request.
- **DELETE** — delete the specified resource.
- **PATCH** — partially update a resource.

**REST API example**

- `GET /tasks` — Retrieve a list of all tasks.
- `POST /tasks` — Create a new task.
- `GET /tasks/<id>` — Get a task by ID.
- `PUT /tasks/<id>` — Update a task by ID.
- `DELETE /tasks/<id>` — Delete a task by ID.

# 4 `requests` Library

The library allows working with REST services:

- Send a request using `get()`, `post()`, `put()`, `patch()`, `delete()` functions.
- The server response is returned as a [requests.Response](https://requests.readthedocs.io/en/latest/api/#requests.Response) object, which allows checking the status code and headers.
- Extract data from the response (for JSON format, as a key-value dictionary).

For logging HTTP traffic you can use the [requests_toolbelt](https://toolbelt.readthedocs.io/en/latest/) library.

In [ ]:
! pip install requests_toolbelt

In [ ]:
import requests
from requests_toolbelt.utils import dump

resp = requests.get('https://api.github.com', headers={'User-Agent':'my-client/1.0'}, timeout=10)
print('headers:', resp.headers)
print('cookies:', resp.cookies.items())
print('text:', resp.text)
print('content:', resp.content)
print('json:', resp.json())
try:
    resp.raise_for_status()
except Exception as e:
    print('raise_for_status ->', type(e), e)

print('HTTP dump:')
print(dump.dump_all(resp).decode('utf-8'))

## 4.1 Testing REST Clients:
- Open REST APIs: https://mixedanalytics.com/blog/list-actually-free-open-no-auth-needed-apis/
- Specialized services that emulate REST operations (mock, stub, fake, test services): https://httpbin.org , https://jsonplaceholder.typicode.com .

For example, GET to https://httpbin.org/get returns a JSON object showing the URL-encoded request parameters, headers including User-Agent, and the client's source IP. This helps developers verify what data their HTTP client sends in GET requests and how the server processes it.

## 4.2 GET Requests

### 4.2.1 GET Request with Parameters

In [ ]:
resp = requests.get("https://httpbin.org/get", params={"name": "Alice", "age": 25})
data = resp.json()
print('status code:', resp.status_code)
print('headers:', resp.headers)
print('data:', data["args"]) # <- dict
print('DUMP:')
print(dump.dump_all(resp).decode('utf-8'))

### 4.2.2 GET Request for a Resource by ID

In [ ]:
resp = requests.get("https://jsonplaceholder.typicode.com/todos/1")
todo = resp.json()
print(f'id={todo["id"]}, title={todo["title"]}, completed={todo["completed"]}')
print('DUMP:')
print(dump.dump_all(resp).decode('utf-8'))

### 4.2.3 GET Request with Filtering

In [ ]:
resp = requests.get("https://jsonplaceholder.typicode.com/posts", params={"userId": 1})
posts = resp.json()
print(len(posts), "posts retrieved")
print("First post title:", posts[0]["title"])
print('DUMP:')
print(dump.dump_all(resp).decode('utf-8'))

### 4.2.4 GET for Related Resources

In [ ]:
person_url = "https://swapi.dev/api/people/1/"
resp = requests.get(person_url)
person = resp.json()
print("Name:", person["name"], ", Birth Year:", person["birth_year"], "Home World:", person["homeworld"])

# Follow the hypermedia link to get homeworld data
home_resp = requests.get(person["homeworld"])
home = home_resp.json()
print("Homeworld:", home["name"])
print('DUMP:')
print(dump.dump_all(resp).decode('utf-8'))

## 4.3 POST Requests

### 4.3.1 POST Request with Form-Encoded Data
Used primarily for submitting HTML forms:

In [ ]:
payload = {"username": "bob", "pass": "secret"}
resp = requests.post("https://httpbin.org/post", data=payload)
result = resp.json()
print(result["form"])
print('DUMP:')
print(dump.dump_all(resp).decode('utf-8'))

### 4.3.2 POST Request with JSON Body
The `json=payload` parameter serializes the `payload` dictionary to a JSON string.

In [ ]:
payload = {"task": "Write code", "done": False}
resp = requests.post("https://httpbin.org/post", json=payload)
result = resp.json()
print(result["json"])
print('DUMP:')
print(dump.dump_all(resp).decode('utf-8'))

### 4.3.3 POST Request to Create a New Resource

In [ ]:
new_todo = {"userId": 1, "title": "Learn REST", "completed": False}
resp = requests.post("https://jsonplaceholder.typicode.com/todos", json=new_todo)
todo_created = resp.json()
print(todo_created)
print('DUMP:')
print(dump.dump_all(resp).decode('utf-8'))

## 4.4 PUT / PATCH Requests

### 4.4.1 PUT Request to Replace a Resource

In [ ]:
todo_10 = requests.get("https://jsonplaceholder.typicode.com/todos/10").json()
print('before:', todo_10)

new_data = {"userId": 1, "title": "Buy milk and eggs", "completed": True}
resp = requests.put("https://jsonplaceholder.typicode.com/todos/10", json=new_data)
print('after:', resp.json())
print('DUMP:')
print(dump.dump_all(resp).decode('utf-8'))


### 4.4.2 PATCH Request for Partial Resource Update:

In [ ]:
todo_11 = requests.get("https://jsonplaceholder.typicode.com/todos/11").json()
print('before:', todo_11)

update = {"completed": False}
resp = requests.patch("https://jsonplaceholder.typicode.com/todos/11", json=update)
print('after:', resp.json())
print('DUMP:')
print(dump.dump_all(resp).decode('utf-8'))


## 4.5 DELETE Requests
### 4.5.1 Request to Delete a Resource

In [ ]:
resp = requests.delete("https://jsonplaceholder.typicode.com/todos/10")
print(resp.json())
print('DUMP:')
print(dump.dump_all(resp).decode('utf-8'))


## 4.6 Raising Exceptions on Status Codes
GET request for a missing resource:

In [ ]:
try:
    resp = requests.get("https://jsonplaceholder.typicode.com/todos/9999")
    print('status code:', resp.status_code)
    resp.raise_for_status()
except requests.HTTPError as e:
    print("Request failed:", e)


## 4.7 `requests.Session()` and TCP Connection Reuse

- `requests.Session()` allows sharing headers, cookies, etc. across a group of requests, emulating an HTTP session.
- All requests within a session reuse a single TCP connection.

In [ ]:
from requests import Session
with Session() as s:
    s.headers.update({'User-Agent':'university-lab/1.0'})
    for i in range(1,6):
        r = s.get(f'https://jsonplaceholder.typicode.com/posts/{i}', timeout=5)
        print(r.status_code, '->', r.url)

## 4.8 Downloading Binary Files
### 4.8.1 Simple Download

In [ ]:
file_url = "https://www.python.org/static/img/python-logo.png"
resp = requests.get(file_url)

with open("python-logo.png", "wb") as file:
    file.write(resp.content)

### 4.8.2 Streaming Download

In [ ]:
import requests
url = 'https://github.com/jsikyoon/OCRL/raw/refs/heads/main/pretrained_encoders/slate.pth'

with requests.get(url, stream=True) as r:
    r.raise_for_status()
    with open('slate.pth', 'wb') as f:
        for chunk in r.iter_content(chunk_size=8192):
            if chunk:
                f.write(chunk)

## 4.9 Uploading Binary Files
- Use the `files` parameter — a tuple of (name, open IO stream for reading).

In [ ]:
import requests
files = {'file1': ('python-logo.png', open('python-logo.png','rb')), 'file2': ('python-logo.png', open('python-logo.png','rb'))}
resp = requests.post('https://httpbin.org/post', files=files)
print('json:', resp.json())